# 도구 선택 에이전트 LoRA SFT 2차 + 시험(gold) 채점 — 한 번에 실행
**L4 또는 A100(20 GiB 이상) GPU**에서 **모두 실행**을 한 번 누르면 학습부터 시험 채점까지 끝납니다. 4bit 양자화는 쓰지 않습니다(BF16).

순서: ① GPU·박스 표시·시험 입력 확인 → ② 학습 데이터 만들기 → ③ Qwen3.5-4B LoRA 학습 → ④ 시험(gold) 채점: 2차 모델 → ⑤ 작은 분류기 비교 → ⑥ 시험 채점: 학습 전 모델 → ⑦ dev240 평가: 2차 모델 → ⑧ 시험 채점: 1차 모델 → ⑨ 시험 채점: 2차 모델의 대조 조건 → ⑩ 표와 결과 ZIP.

- **Drive에 이미 끝난 단계는 다시 하지 않습니다.** 2차 학습 데이터와 학습된 어댑터가 Drive에 있으면(2026-11-17 실행) ②와 ③은 몇 초 만에 지나가고 바로 채점으로 갑니다.
- 중요한 것부터 돌립니다. 중간에 끊겨도 끝난 단계의 결과는 Drive에 남고, 같은 노트북을 다시 **모두 실행**하면 이어서 합니다.
- **결과 ZIP은 단계가 하나 끝날 때마다 Drive에 다시 씁니다.** 끝까지 못 가도 그때까지의 ZIP이 `lvlm_insplad_results`에 있습니다. ZIP에는 모델이 낸 답 원문이 전부 들어 있어, 표에 없는 숫자도 나중에 GPU 없이 다시 계산할 수 있습니다.
- 모든 단계의 출력은 Drive의 `agent_sft_gold_sft_v2_human/logs`에도 남습니다. 화면이 닫혀 오류 글이 사라져도 ZIP 안의 로그로 원인을 볼 수 있습니다.
- ⑤ 작은 분류기 비교는 같은 학습 사진 660장과 같은 라벨로 작은 전용 분류기(DINOv2 특징 + 선형 분류기)를 학습해, 같은 시험 문제의 Q1·Q2를 풀게 합니다. "LVLM 대신 작은 분류기를 쓰면?"에 답하기 위한 것이고, 위치(Q3)는 답하지 못합니다. 설정은 고르지 않고 기본값 하나로 한 번만 학습합니다.
- ⑨ 대조 조건은 "도구가 도움이 됐다"는 결과가 나올 때 필요합니다. 가운데를 늘 같은 크기로 확대해 준 경우, 도구 없이 같은 횟수만큼 다시 물은 경우와 비교합니다.
- 첫 셀이 몇 초 안에 GPU를 확인합니다. L4가 아니면 지금 잡힌 GPU 이름을 알려 주고 멈춥니다.
- 학습을 시작하기 전에 시험 채점에 쓸 파일을 끝까지 점검합니다. 문제가 있으면 학습 시간을 쓰기 전에 멈춥니다.
- 시험 채점 규칙은 실행 전에 고정했습니다(`docs/agent_sft_gold_eval.md`). gold로 설정을 고르지 않고, 돌린 조건은 전부 보고합니다.
  - 주 결과: 2차 모델, 한 번에 답 조건, 학습 사진과 같은 드론 사진을 뺀 gold 사진.
  - 위치: 사람 박스가 있는 결함 133장에서 답한 칸이 맞는지 채점하고, "늘 가운데"라고 답한 경우와 비교합니다.
- 2차 학습이 1차와 다른 점: 확대 박스와 위치 답을 **사람이 그린 박스**에서 가져오고, polymer 절연체 녹 사진 60장을 학습에 넣습니다.
- 설정 셀의 `RUN_CLASSIFIER_BASELINE`, `RUN_GOLD_BASE`, `RUN_DEV240`, `RUN_GOLD_SFT_V1`, `RUN_GOLD_CONTROLS`를 `False`로 바꾸면 그 단계를 건너뜁니다. 건너뛴 단계는 표에 없습니다.
- dev240 평가는 새 폴더에 두 조건으로 돌립니다. 11/17에 네 조건으로 돌리다 끊긴 실행은 건드리지 않고, 그때 저장된 답을 결과 ZIP에 같이 넣습니다.
- 다른 버전의 노트북으로 시작한 시험 채점이 Drive에 있으면 그 폴더는 그대로 두고 새 폴더에서 다시 합니다(그 폴더도 ZIP에 들어갑니다).

예상 시간(L4, 1차 실측에서 어림한 값이고 아직 재지 않았습니다): 시험 채점 2차 모델 1.5시간, 작은 분류기 15분, 학습 전 모델 1시간 이내, dev240 1.5시간, 시험 채점 1차 모델 1.5시간, 대조 조건 1시간. 학습이 Drive에 끝나 있으면 합계 6~7시간, 학습부터 하면 6~7시간이 더 듭니다.


In [ ]:
#@title 1. 실행 설정 (수정 불필요)
EXECUTION_COMMIT = "e548776dd70386c0e59467535fd46c13d77015ac"
SOURCE_ZIP_SHA256 = "b62ad0f40ef847d98f0865fb8e03b8f6486bc5efbcecf106ac2836f0355b0421"
SOURCE_ZIP_B64 = "..."
N_IMAGES = 240
PER_CLASS = 60
EPOCHS = 1
LEARNING_RATE = 1e-4
RUN_TAG = "sft_v2_human"
DEV_FOLDER = "lvlm_insplad_qa/gold_v2_dev"
SUP_ZIP_PATH = ""  # 자동 탐색이 모호할 때만 Drive의 절대 경로 입력
UNSUP_ZIP_PATH = ""
LABEL_RUNS = {"sft_train_location_annotation_sft_v1": 300, "sft_train_location_annotation_sft_v2_polymer": 60}
FIRST_LABELS_SHA256 = "f4c9ce51e23c8fb6050dac00462f80e821b2b85c3f2edb0c68b540839c607270"
GOLD_QA_FOLDER = "lvlm_insplad_qa/gold_v2_final"
LOC_QA_SHA256 = "f5a6b5c1e0b9d7061da729b4b5186882e10dca00fb94a4de998b798e9cf1e5a5"
LOC_QA_B64 = "eNrtfVuP20iy5vv+CqOfxzMZl8zIPC+LaTcW2MXu2y724eCA4NVdO7bLU2X3mT6D8983yaQoMl2UVCpSSrbUGGDkEplJKRRfxuWLiH/+VD4+1dlD9dO/vOtevlfaVLYuS/9/yorWP/3p3U/583P9LSvzb/XHx6ff20s/fvJ/e//w5fn7p/zb41N70cPn/GPdvlfVTV1+y56/f62ffnt4rqu/RFf/5bf8018+Pzw/P3z5+L7Mv/7lvz1+e3x+p+x7wPeoUGW//I//nikS6/es6oc8e3h+/JRXj0/Zbw/V02MGjPjn//f1Y7vt9y/7jbLhGeIdv9XP33ZbvnXHL49Pn/NP2ZP/mE/1l7I+sOtT/vDlLx8fH6t+Q1LvAUYbspKZDVF42DD/8ug3/D37+K3dJPzr8ftz+1b/XYd3Rl9pJ5B5ob7/O31uL/mWP/+tvaj+x9dP/lHz4lOdfS7/nrfv/f27/84eHr9k337/Opbrp8cy//TwH3n73vi69pL//av/St49PL/L34Wr3z18efft1/rdt/zpY/3t3dfHf6+f3n96+FK/635Tf373f3d3tFf197Q7fKur6N7uW/6v7Y6PX9v9nv2G//zpr+22/+uhqj7V758ePv767d1T/dG/++f2wp/bN/9nt2dZf/lWP43f/NC++X++fm0fqG4m9/3SvvVhesd/doJ49mu1b/7y0/DP7Fv9j28v3OAvGH9VQUb/7H89D//hf61F8fiP+jn7x+///qt/61//Vf0ZCEWZP71Tf9YWnG1fAAIDtq+QBa35tz+98xc6Jei4/auySMMri90rbP/jf/OXevHnT891Fh6q+z10T9k+XZE/Pzx3v5v8y3f/g/ZPk+Vfvjx+G0Tb/+ux+8jdN/D1yf+Oy8dP3a/98VOV/YbtDyL7/Dmv2jue6t8e6n/Pnv1d3593F/30n//lnz/ijFGki6okW1XEFi+HMwjvQY2UUJOb03oDdgmcOW/HLeJMJNRbwpmXoeTD6L4XYSbcNl1yijV/fQFr5p7k1YhjrNWoW9BgAaT2BSmtpcMRAmusC4ijNQJBhzMdLFkTrgb2SCUvY033eAlAjZPa1rYwFVSmKuGCJg1FBoaBGT10AItYNOdsuEWgiUT6BwOal0Bhas/88N6HQ7bO1QwaNiwmWC9GnO7wA0Br3Rk5QIYcJW6nNMTQ1FJDUzTaqSuCB/GMbnkMXwU8Ttlwi+ARifQPBh5zIDBCjxetlBlguR52aGLpfCGlvG8TXBxD0lkroLz9kTZyoOGy5BybGjXWZXVB5JCpIsOsIgPRIh7OeTtuEDtiod4Sdsw5HR9eVvg3uDdzj/F6DEEyqgMRg7Z/BQYH38VpZV9GkU/tI1wfREgVtWas/GHlSKqrhmNlPjgK64RjT9lxgyASC/UPBiIfDsLHi7ZHgIkZx+Ulw+Q4hixhgVgmChaI8Z5KgAzkYIA4BYZDbARBjOkiJpoHlIHe29HG35y2pUKq0UULiig1mlJfM0DCs/GKdXycUzbcJMRMRXobEHMoNnIg1fMSMl0EYAyzERuCrkQmhFcFyPWBEkfcQ4z3d5QKQVliSyZcyYb6QIq3YzBxkCkKZwrSUtjaWJ1fE2RoVud5HZA5YcMtgkwk0nsgJcWksrdJ+qQykXdfQyDFOBOCK4AoOmCMcgwQ0EhrCqlkpV2faAb//wiJY0xj2SkUqMpaRNnrJZUNqTnfRbOskVQ+bcctokwk1D9mUjmGi58P5IE+HIrGXA9nxHk7pMv1aOqTykosSUAc560W6I0ZyxjghUXESngfjKKNhHapacjq2oNooY2uisvhDLip1ivBOSWkJVDmnP02iTFTgd4Qxnx4mRn3sn3zy2ywZgoxP68DMSSOTAcX5Ly8MMCFxhB8AWe9F5U2cmhVqLypbS05C+urhlr0rFNi1vGCTthwg9gRi/SmormHYi0z4dyrRVu8MeLxozM7rNMhnKus63m23kjWfbDFY4sLDBXWHDwiZfzvqSfkEopLHGXEqrzABp1SpmB9TftkLr6Keh375Ph+W8SYSKB/SL5b+fq8UXJOEHlfNUR0GQfWigcb21soRlKPoRhDNeXSaMMERl+SmK8j5dKzEQ3vNS4SQzlrxw3iRyzUPyR+bD+EQlq0CfU//qcB3Su0FrhDFBT0Fkzi6FHVmqsc0ZYsBco1y3pgvsgG1inrOWXHLaJHJNSbi8DOVA9+OMRXORgf+bASfrQ1OhACq071jg0A95zZ7lVfziNgQ7jVWywmhGgDHS7kk0kkXCgKd7x9NP3iMNQCASqjE2e3mCb33hw7xLyoAK6ZeEYzR6hfxp45a8MtAlIk0ns98wmVhlfh8OOuThlRRPcBlGDNaGXQpg0dolEz5WRd1aAivmIcxczmXVYJoxzfboOwEYvzXp58tPDnOH//JeSYcbrOqE42dhd19TZJsGK0lUB6E+NdpJcB5Pv+Aa6PIs5oXTlxUhJUFV/QJwIzVTSl5hn1bhGf6LwdN4glsVD/kNy3l6Hk3IzxIRRZKWGsNXoI6dyctnlKyPmIx/7g8BiN3HdUMTJ4N0PoBUisdSOCbWD7W9HOTFomeExyPUo5k3iEJhfJWUshjpmqazZeYdKzjDWhNSI0p+24QTSKhXpL+aGjlk25ZGni3KO8GprQOjbU55aF+riLsz30aPKSPGDcXB9IClVKIwqoqFVTVM0VQ7161sgw6FYJ9Z604waBJBbq7SSKDqWZD+LIi27XcR9psRZOQtTTWrT1ANJzaL2j2xszDmzS7ZmKAlDVlclzakqTX7A9U6xkwLPkEbdMifN5O24RRyKh3klxx/2jA4y4n9cEEKs8agQ6PliRfXXzsfrlVACkFFNAYwomEYT6moaInjcL9DqGyCk7bhFAIqHeUm3hpij5VtkQmN0XJiNq26ecE88Nl5QTVXnTWCyhafQVyW7s5qlnZhWy20k7bhA6YqHe26scJLq9ACsXoeOTQ6NDbthpdn2P611oVrpywbTBgyuV15V2NRWFJrpigzc7yzwjVKs0eDtpxy2CRyTU2wCPgzS3Q/nhE7huv6za4I3aXvctYlhAhBDzaBltwY0hZO2SbvBW1oqLSsiWVY02v2LLAm3m6au0SjrmtB23CCKRUO/Oy0EQOSGCulZ+mNGEZktkd53blHj7N6RzIX3CfQ1WSVMo13inC+maWRiYnZ5j1DpZmJN23CB+xEK9tzx5I9/+wAyN+Clen8gFwxJI90yO+24mtp/FA96NmRnF0+6cAIC4JtcV1arW2Eh9TQCheXVepiXBeTtuEUAiod69mDelX1aKgGjDfcHNPvGiVN/XRDlR3A/fEdC7ght/TwitKmdcYJ8px96MCVeSBgUSehy0PSV7MMI+qqLacT9pGzRNBaiYmyp31DijrolH8yQPWCebc9KOG8SjWKg3NK/nw6snD14vmcNOY+hZ0M4hDNCE3n4Jva+RNDqd+Lge3Q2gA+ScpZkLp3xqJfWl/ak+PVbvn78/f62/PPdPdwBG5m7r8OTpu9dw5d4r0+oX7MKkmLEcaSo7u2wHGqesO48I82vH0BDXDmoxWX99/ph9zZ/yzN/ymGl5LSy0n+CHgaRTMd3pISdU4bzaw1kAERD7bveaILzU5JQJL8WbGLt6Yu36yTxOie0vVUjc/7XNNho7VB+HiMvuMhC9v0e8x8TtDruFvckT3kK128Njk2azyzH732y7jer/65tMkpjdjeB21+6evI0VO+vvYhf+OxnV/OHqf5pV9lx/zPzrb3mrbpfDN0ASsrYpdW5UU7hL4xtMcchLJDvGjDsF3g4uuzl0i4V0a91WXtcu4TgPf61uCWYAFLQDfqE4t0Mtox3ICLdIBiTZYYsS3gNi+7MCgeF63qOUHgDU/8r1CBedVWxGdwxIaEbrevzdw1m8Cw0PJbCHyMkuxhF1n7C/Y/9x7XAJGzVArbLQI+wPH0TaqSj9PUCiho/VMoFGe/Duejt8DFAoo4/hrCFsP+1mwFcrcrWCsqgbWyq8snGptM1Ym8WNy2jd7cFvJKYbip4fnUqQ0GxY2kMH78FRm2D9BRSizkjbmYKk9wZgh38Bd/YWIoDBAIE72BpQfX+RohFMQmuY4ghLd0bs6HLCEciFdfoHCnd2MGpxb/WOTGRliAPk9a3PR59ut6gRkQGqHSPZbaGigboqbWnyBvKc6dKoaKfo5aWV+RPx7ah4eN3toWIkpjsrMs2hc3trzu7hRe0svn4KrgoI08PQACl2b2spq0aOuCUYXY/DFrA3aJXdwxyIHYOiGdByhF7D37zZN0Y4xL3fvb9c65HJCci9Vb0ViCubstZegQrrH13xhSEO9BSKkCnzZ8abIe7IutuDuEhMN9ST42BXsJn0xPUsvx3M7PAJ9uYWOIN2BG68+/seP2Bi8nlnfeza8oA+I2fd//z0CA1F9/HM3uYbAHcwKXkMjBqDebZDz8FWs6O4Io9s2JYGMlof9593d+doEXHa9lbuRtCQvIKpvFT+49RQU31lgw/FG2YLoOGRdTeHhrGYbicKeQYF9Xo5V9w7ixigwbnOu+1TEyGGhr1Z1RtkO1NwlJdpA5E7nOmjgxvBE+/yQ865bRoHrskvndMAFYW/vFIebRtyinV1eN3N4Uksplvqnvh6Esdx/3EtSjsMThe8YPIETqrHlsGg2V1Le+PE7T1EqyBEvDYCJqalPILJqXFOCnvxGD1OlZ5ZZSgLZEgPr7s5MInFdEsE1TNmZl2Locr7ENTY1Bg5U3EAfZSntONE4ShgH7tHeybHPrWox3e09FXGLZk0pvYIy5AjcF1XVX7lgBGwy0Z8zsUCRtG620OhSEw37yIdihQd7ROwloO0h6Adl0IMDuAgzmODhb1rNLyD43zhOHINbkyfMKP1R3k7O7qbSQ3xqAGtRoFutadBAHuraxyP2i8De2vM7jOfLjA+dmy3XfBqTJzbpwFYHIzIcSOMdHuzT41i7W32sn3iHc/N/ZhLdfvs5vhGNi5c09Pq+tmIZrh6FP2apEj3H2kjYC21FKZGXbi6AIN8Zf+TUDJjlvc/o3U3B9axmG7DZDzK6EiNMUz79N8ITeyYBkbkaATZegyRLyOwtwbtOMc4wONoj6nbCsAdJvG2+L1W1bmHYYsmL1Qu1ZX5veDUSHkX4/dOl90cEsVCujuvaQ585oGTZQdTZURYFekdzl3p5GCWychAw1GST/f/2plCw2V2bzaNro+oZSNixGBEkh7lMq3yllP3fJuBq6KuwJIz5OG5KNSV4UqDyrxMF8eraN3tAVYkpj8kMeJAQcKrxx5dZVbaYMiMqf5GOhdvV9xkAhBBTyDoDah9yG0fZWv5/Huocz3FondKd0bWCPMUMba21N5TfIm/avY22WgvLTLi104KB/Z+t9szZF/+VDwQIfqHHJ7N8cgFtuN/oe1dz/6eUJox+KVAoxI0t6O49Xg8SpQMbrdjDggcHox6vtpmADmvrValysm/YD03f/dyJQoWMja4fInCdN3NAXIsptsB5A+vH1x5tVTqnrg1So6O43F2KFf4MfkxSpdMw2NiJzUEL5YQTIJ+kdlJeh+BG6VkcP+McYGYlhHhd88i7osrQj3sALp7Zt3IY/fO9P6kAOPcprIxhdKadVUT5BXUfO2csD/vMrDL54SjdTcHirGY7l0TDzduvtpUPd470XaEOoM/HeDFjfMUQ7mA3du2RmgEeRrsCLJwn9nYQ9ZgIAZOnAfS/Q57csse4faIpUWPy3JHSe39SxIzosiI3hLClYYdlEXjYV4KYndls481Z1q7xc2+aN3NIVwsplsa03e8EisdVxzHBtMAcGMbyKqJ4UcjW2mEiOOSKh2XEQxp6hFoTuw+a60Zl/mPQHRAzUma2rIz4+rXvTXqhuvHdQpRxb4ewfrIshz3WlG8iyNsBBgr4sLleaNqBbl/8GsDI4oHsOX94WjdzQFjLKZ7v8o1ZpcugYx7+kvnDSs1KqYXYR7TU4b42piJ03awHBNS7Ise63Sbdp0RwqKCSZeVIZkyyh7jxLjcxz+3glulGCyoLUSzpSnyq7us2ruWagWXdbLu9nArEtMt4dbra+qvh1p7T3DnSvI++h/D1igpMlD2JgWkP9ScDu/QhHi3D/o5lr5+ayv405SQAzPbsnKFyLXxhz1OOFoef6brbg9/IjHd3Ljm1/fSvEpqd3DEcMCHERFFaxkXRIwK1/edKwH0mPXGdtTxUuHOp4zLvkZF8wIwycsOBJS9czdKIXijTnccvaFB5g+5BzUTx/Objg28IaA35BX2H493tf59Ibzqy/I3ApI1SQ4lQVE3ktdgrgyShlw7NnpxkIzW3RxIxmK6pbzC2VMhr9IX5MfMoxLs2nMMzX1DLG1XHbZrNflDgw3VlojswVPThCYyave5v4WJsbPxeuQyAZSGYrWtgFJljFRSc4vDeXPtanoGlxlrF69miNbdHihFYrqhiQgHSho+nNuMba0WwTQYNjBi5VmEIa6PZtxLiEcpxJFR5MCNoSVMlzPj1sKjENloJ41G7S2nXafgjSCRQrRN3pRklf9wzQw5+Ovjp98/e+kOU0vef+9+O8+/5uXfPtVHAOnI3Qe6fBBnBo6kKI+tfqDZx3T5eXg6ukWMUhiNcAGy2ef8y8OnX0czXLoP8+Bf+N+pWWJeQyTKW6q9OthU7dx6/Zfwam7FM4biiu6DUB6owuwW79sZuwtezYyA6n5515/h4kHTVaArb3txrZvqqrgRZ9oQVMZ2Mdw4svzWcSMW5S0RHmaCUMeG0X14bc3mcrDhnNoxA5zoML6SvRUccENbY13awGHYKW2blkBY5ZqvCxxRDRGRzkCpxYDj8PKbB45IlLfUbOzsEbhXcJC4tQipL98JXo3RWvqiGmXMrsePWNMZIoTedQ9Eem+ddMhiujfTnisHeY51TlzltWIpr2yT/NCplDOyy0HL4eU3Dy2RKO99Ud/WF/WXdV0ZTWT6HmJgaEctZ9M3oUdrkrZJELmpnPLmlCXWSNcFjiizg44z5OWcmcPLbx04YlHenk1Svh5APry2pfJi0EGtJdJzDpF1GGZLIYOkHDtLaQNHLXmtcyq9oaShwaScGc2SgXZrOTPR8psHjkiU93mQb6v6WBU2vFtiEPoRigS4I8+4kHpWAsb2ZJh2TkR43zk22NfGSkjvoGbj0o6XEDssyD96WeXepnJXhZheP+k9qLF+wkH9ZAVLYdBb9986SMU/hntC+m1TE9fN8DB6G7QfIAvSt9jTjnasP4G0kSfPS/HwKNBQrS0lZdwoJRmvZdpMFt88ZkRivL30zuuJdecNFlwOOSwGV6gd82t75FCm/1vb+C7teArXeVOU2ricxR9UaSEHCmVWrwUd09W3jh2xIG9oSvMJwZQFGyEtF1DRQ65HtS3nunQOKIujbE/K0KGlsaXN2Z9URaGLIinoAAUZ0lrQMV1969ARC/LmXJVzKGnnpYiXszuAjcY+Gguh6huM5f5VO5Eq7WCJMbVDZVjAOSKrksoAt/0QCVfLAEfLbx0/YlHeWWkntJw4r4v3kpxWUiHi4S0OCQx6ZXf9I6xGTtv6EKgb7QxKkUOlIE8sn2MzMGq9fM5k+a3jRyzKG7I/DtUTpswjaVtdUZ8MZhsYJWBb5lnouC2K0zY/LNSKK+sd5cLoQnRaEVPBDHC1kOlk9a2DRyzIW6pGPj5C5MW4x9VJaFp76yMQ443mYHu0PWNC+NQZw5w2eJTeOmq0ybUUJoe0wKNlibn1gqaT1TcPHpEgbyhoerbjcnX0MGIDecR7LgwdjpC3PJyEQbeceNjUllKRSCVNJaVLrKoGvWchq4VNp6tvHzymgrwX8aZsdJBYg73RoSiYGsiaQxAEDCmwSeNGnhdYGPHHlbeeZG4UzdUytZhZXi9TO15967gRC/KGjI4PZydayqtxw1A5CzsCq1H9SCzRAUK0Aki8/j8vauOoJnG2NtbWSQFHO4IAeLVIabT85qEjEuWdIHbM7HiRO3bBPAtZC33BnTE7b0URBm8FrJXE3ZVCleAIyoqausbU6masZGDManmW6fJbR49YlLdXcHdmnuUQesw7LT+s+GrwEOXNQ+rn0fUF/wjKSl8lYwQP8dLDEyQAIVBXjQHVKG/zNhUnUBcDblqXopw6oH+4cFXMubtvHoCiH8ItAdB2461e9rsCPjSm70qCtm8ZYK1JO25SmqJx2pm8qZry6ska/LHNEGqzVseAaPmt40csyjvH/a2dp1cu/fUGSt8KH7Ev+NXiAm8VWCTxngGVlRqrStdoVIXaJdU5kRRm3ipcq3NitPzWoSMW5e1MCD9EMfuQqMnRuTYBOFRfEIMAwD25Xfu/JQ0ctbM6b9AxSyPaQWKdE00GyxX6H1l+68ARi/J2mo0c8lgSBQ4iZty1N+vH9xC2XPauMSKIKJ00cDQmN6Y07H9t0mibGC3Va7ZdjZU6XnzroBGL8Z6neVtjs3VLYdrJEqHsBclDfOhQpBgCM8SfZSZtOmrTaN3kyFBpTZpMWs1UmTNasCfi4eU3DxyRKG9wov3rRxAenQ6xcoYXe8dEMRnbV9KJsT18HEnSpNDmvbSVVYC6JjsbIP0tf3p4//Hp4esRoBiu6yCheHiq3n9ptRfiEnrE7AhLfb9Up//Hl5nX/tFSP4ypgumYKiU2ay/P2suzPbScqtXDR/6hBfv0a76lQpVDrkTCg1/QtfyM4E44ZTqF9hre51/bHIhOfIKD1ALMVVUTGmXNOqrdn5dmel6iwZEeOVCvV/ZXLpy2+kei+EOq/xnNjg/mLo4Xyb+k/3NLnnG08468BUyhV49pp9X1c861ZnlZ/xNhXpBtdANii8Jig/VKR/sPNrnVGTl4vb4fWShp/Y6/6pvLTS7r8c8nJuN9Xq3VipDsbsYKhgbmWpxV3d9YDFs5xKpqHyAB1a4a7yCWJSoqXV3RZax25UxGb7faJ8ukrdbR13xDvSkOjjc53NXmtT31FpicpIzTRk9pShbQBNfcoCjHac9EorrQ3jEsaiapSiVrGusgkU1t3Vh9xJ5rrJ+8cNpaH4nihrT+fGP9hcD/BclGaMiGGYuKhEMYrj3euzoLD+E68SA+NapEA9BUhRMo7KqueqykDsYeNZuzXfVTF05b+yNR3LX/hGGJr/TRlzjxAYdZZtqpLibnjCXXoYCgI5v2ga+rivPc6iKnskFZ9cBHPdVMbWikO5bxbJU/deGkVT4Wxb2L1CKt91eNzmlQErx3Ut6j7PSfLYANHr2xOFdamUh0zgBDVZaFQlNBo+CaR76sdeTLJvQ/FsXtVSaV5w4zfW1p9WL6z0YL7KLz0MfxiDiMSzbe4HdpR+ct2cZoagrUTVXmcpnovDKSEei3R+ejhZLW7/irvjd6e5NFf0Cv3x6e7zS6U2ZrbagVEmVhTpnTCMdbKaEpnLJUOba8ki6DjlQQJBM8I7N+ZKG0dTn6qu/jPw/p8gkR+bX8865sR1kXmHDOOTDd0dx2nse0XfOiAmcMmdKIUbkuVo3Fu4hoymqkN1rOJs6cvHDS6h6L4s6lP16Bc4VgnDK6nwXOIMEEF///YRCeWKUZUlf5HAslBoqSGuUurPJuopm4nMrPLJy4yk9FcefKnZB6v16d/95it8qoUErDJvm+iCWX5AAAOc9L0HJRcizocZRMucXIsXMLJ63wsSjuLVHf1kl5LY6Ntv5sD63XjeHulSNrMWTcuGXepH3Il94bsdxUqkBbEuXXJMTDWoT4bRBmY1HcXsrt0LDrAwNrz2PDLxSXU9yXuBmlQAcEsMbb+SkH58qyJlc0rm0bVajGXDLRpo2MlRMX49bMLpy21keiuD1v/jC17kW9v15tvHLEoRGP1/zwypKywQjQAgrT7qdR1izK1mVZKnL+s1zWxueJKS7L2fgzC6et+ZEo7jb+0dH017DxvYGvrfQqb0Mgz6IicMHB95LTadv4tcrBQY65NMgury+UimPKBHmBVNx0oaR1Ov6qb2kE0hrV7H9dR6fRGDFmd4wbCM2wWLu+Skar1KmyNZD/X6ELMhprvChVFtXY0LZGLUWVnV04bZ2PRHEPzr+1Ce8vK+bfW4Xv9F0cAkEfp3OWU9d3rivL6J2NxpY2v2QtHKmxec1WLVULN7tw4vo+FcWtNcAsz5hZdhV1twb6NnZKKFTDgOIwaMjrPFHiCfhaCpvrdkpSrSoL1SV9dYhC6HopX3124bR1PhLFbdj1Wy193R30Wnfnu2PVD0jWbfNKSrxTdl1wXWBFyiouylWVniL7W8bkGDHnB+hOXThtpY9EcTst8j+8PiR/vQCd8iodSlz9K4supOAAbJ+ON9omHqBrVNN4FwRNRcCi1mpdo+JOsToz+oyk+5GFktbp+Ku+HeP9YDfZtw4c/nnlulYIOfY2wx4aWdjdvGGHNDdvK5GqtkZTzrpUTE2tcmMvyqPVMKG7msV4tHMLp639kShurpHFqxk21x5goXbnehh6I2hFBT5t0la8kkYMlTkXtXNQL8qdH83NbO3sydxM1hM7+3W0+XMWTkLdX+gbHwvg3r8izSoZtDYk0hV7efW1Mc5w13rSamdI0rbdVe20I6tz4cI/eX4pTceJR/3KDlXnLJyspkcCuKXQ3Jnju8vX0ucW0HQCdNJZ7Awadaff5M34QJpvm1SYxL108J4GeKuxrHOp0arFNT32q73Vk5lXtok/ZaFUNTn+gu917Gnm0VErGzLn7QzK0JCCWZCDa+5gdsBLMqrsVFU5/zkM5XWl+GLmuZ1Y0Xo583xm4WRVPRLA7Znnq7SXWrehrDMWAvedLIYeFuydq9A6XjNbk/ZcJ7Q5cl2K06bOjZPlD/CIuQr+3BWGMw7wwwulqtXxF3zzpvgbI2trsV9Jef3tzu+2J1ynyGSlH9pGZLRK3BSnnAmsQ9HGmmrZWQ+HjlmZRMFYLXZ+zy2cqqbHArh3hD6hf9xBXV8pwEasQnwNQVSInJM4K93fyOHs4PhUVJ1dU4D2T6wr0KqsL3BoSybklji0JwulqsrxF3xLnV5fX4d2NFD+81pnNgTiKlhFwc9WwhSCZ46VTlyNtWPJnYJC1SIK5VInNuHkYHWLndhzC6eq5rEA7lUqiUbXvJ6bMFkRxUjvXGsigd0EF514dM2oJgcDJdvS1IWGxXVdxTOPDWZ8RjDt8DqpanL89d4OLXXGnU6z/AS1ZtX51mhBhWpT1gQS6CtMKC5tPRYoMK8actQ41qAuFiUfD0wRvdyZPbdwqpoeC+CGvOwzmrRfi8OCzBD8aSTNEoamOq/dQzoMEj+wRefKkoK6rspSLFzMOJ9krfRy6bC5hZNV9EgAN2ScfzijmvS8RNgS0TRlSHVqDS3lqKezGOf6FBhB4m64rZpKudwiSiVO45V4qbAWLzX5FFksgHvi+4TE9wkjG35ZswCFtLKuZ6myt+W7djEsCJ3aG/LeWNqDlVxVOVsrpZUpTIX6AtQ1lxnAJahrk4VSVev4C76NzPfBkrITFPoajZ/ASO+UqyH1bXR4xcCoEm/8VFJeo7GqRtfYyi1PXVNuqoFsTKbNGU74kYVSVeX4C75XjrxtNNo8J22BylDXtm4K9rgOFWJgyYQZ56i9hh+auPA+kdPZHx1G6qqAKi9LlTcXUGnJtMASKj1ZKFWVjr/gW+qxfELZd0rMcosmsFUIcRc/0+HE1qrlmqZ9OleuIvDnBxj2v7fmYtVgDJOAlyznX88snK6qTwVwO8mxc3hpxwNpa8XMLVLwqNF45zrY4aKJd7kxnXhurMa8bAB1jSKVQnMhLrlaikuePMM0/oJvqZNyucYIpLXqOklBmIMAwiaUhbFX3/A3ttqo1FXZUV5ow2igqPPLkclpErrWvFz2a2bhZFU9EsAtedwHBpoeCqBdxTi3XsEDb9wY2+e5vYIHspoGFEo8+1XnJQkoQ6VzBRXXss7tWta5TV7TIwHcHAn1jNDaNapGUMDpPs8tJD13zfZpbjQq9QLvRoHVZa1LZQCdq69U4G3WKvBOvitLLIBbOtNPCLm9fqrZWk3StQPQfQSdg/ludr6513Ov/om3cmCpKy11bcnmDc1w135//Fv9/vn789f6y3P/UAf0Pbp6nnhOGjJ/nB3W8ni1ef55tNy8bv+wZKzhYKYazoqy/HP+9JRnRf4p/1I+fM68WuolWjlEArg9630bsXVy4EKuDJG07c70tp2q7l5R22otcU0v/C8MuQH0/2lu1tX0ONMlOtOWztb0w8ttRNMjAdxSGu1s8/1QcH3VSaVkTd+lSTndvwILITfOLN6mTLpbSztGPTdY17UYh6q+8MGuMzg2kPw1B/tkuW2oeyyAOxEmSQtes1bBgmcxYIItj0qbLpcG/geSetJcl0aDSGW8AanL/LKaDpYyvZgFP11tG3oef/03NIf4/IFmxw/3l9R9xl94tc4LOBPGH0hQ+LYuJTSDUC0tfSY692m/fQJa35RSVY0HqrxEye1ltZ45I7uY1k9W24jWR1//LRnz64w3WtWaZ3YGgjVPyob2yUBGKDSFQbSESZvzpuK8caoEdHXuyuai6o4GM9FLqft0tW2oe/z13+k0J6j75etTyMnuRPcuexhHjuT6mjPtX6YejxdDjC63XFpDurzssY6gMrOcnk9W24aex1//fb7ZoRDd8Tq0tdoy+sN8d3TvmsMA9oUr5kDhSjJqLqapxORkOC8qW1xUzQ1iBoJL6Xm03EYUPRLALR3oBxpALWm8L9J+lcT1qTZUPXsOLIb5RkykEh9vZFGUEIBrCEviddPrP1DVjcqEz3bTjyy3DT2PBXB7SbcXtf3sgpZ1vXTnPfJ+RAKGvsvQIkBIvlH7X9JOurVUqlopyW1jLMiFU+ycaQvLpdgny21E2yMB3JK2n+2sH238tm6enbW1ITKH2I8pJcUY4vPaG/om7cic44a5sVKZyv/yGr2u0mPU38VhhorPVvrDy21D6WMB3Lu/vUnbP8y3gooXPKMNO3Gw3dtuojyk2MM4FSFLczOQ2q2vr+sFFKUrc6NZYS4WL0yq8X62kuVINZPltqHrsQBuKNn+YY2q1pUaPTIBS4jDO9MPRgIcOr4Jik3ccS/INpXoikxTSlnyFTTdLqvpG3PcYwHch6EdmrRwnE8z30fqzae6EvC2e6ftvdPuXHDjCUTgEE/2fSLnurTk/iInhlxAX/ZcZ+YMaDGybLTcRrQ9EsC9VXuaUxnYn939ce6cCgPJW7fdhtC8Ekq9iL0QTWVtNXtgKqqGL8+bk0V5c1sz4KOv/5bGpW2rht04DIw5gF1vSGrbyoUUO/m/Jt7ptS5yY03diCtslZeXTb0JHu8j94rU23S5bah6LIAbbBt5INN+rh2/bgLOWMEQnwOldaf+6MhxKIux3oVPOxhfFUWRl4pz4Jo9Rl3WbbdtmRov5rZPl9uG0scCuKWqt9dH6E6YzLDWQFRtOJzuDnYv/IkfeDVg2aTei8o7H4YrZyw1db7y6f5yPapatrxVbUvRYwHcGfFvrXj7sGqaHdCEklbvNmoMxHgk4L53jUmcWlNXTYmFK0pbllyY5sKEWZOB2OUIs5PlNqLvkQBuyXE/QJs9nmovXxumW07n0TLZQK0xjGFuixPqT3lia1JS+v8PpvsEnQ=="
RUN_CLASSIFIER_BASELINE = True  # 같은 라벨로 학습한 작은 분류기(DINOv2 특징 + 선형 분류기) 비교
RUN_GOLD_BASE = True      # 학습 전 모델의 시험 채점
RUN_DEV240 = True         # 2차 모델의 dev240 평가
RUN_GOLD_SFT_V1 = True    # Drive에 1차 어댑터가 있으면 1차 모델의 시험 채점
RUN_GOLD_CONTROLS = True  # 2차 모델 시험 채점의 대조 조건(가운데 고정 확대, 같은 횟수 다시 묻기)


In [ ]:
def extract_source_bundle(encoded, expected_sha, destination, commit):
    """Colab bootstrap: validate all bytes and refuse changed existing source."""
    import base64
    import hashlib
    import io
    import json
    from pathlib import Path
    import zipfile
    payload = base64.b64decode(encoded, validate=False)
    if hashlib.sha256(payload).hexdigest() != expected_sha:
        raise ValueError("embedded source archive hash mismatch")
    root = Path(destination).resolve()
    with zipfile.ZipFile(io.BytesIO(payload)) as archive:
        names = archive.namelist()
        if len(names) != len(set(names)):
            raise ValueError("duplicate source member")
        receipt = json.loads(archive.read(".agent_pilot_source.json"))
        if receipt["commit"] != commit or set(names) != set(receipt["files"]) | {".agent_pilot_source.json"}:
            raise ValueError("source receipt/commit mismatch")
        for name in names:
            target = (root / name).resolve()
            if not target.is_relative_to(root) or "\\" in name:
                raise ValueError("unsafe source member")
            data = archive.read(name)
            if name in receipt["files"] and hashlib.sha256(data).hexdigest() != receipt["files"][name]:
                raise ValueError("source member hash mismatch")
            if target.exists() and target.read_bytes() != data:
                raise ValueError("existing source changed; use a fresh runtime")
        for name in names:
            target = root / name
            target.parent.mkdir(parents=True, exist_ok=True)
            if not target.exists():
                target.write_bytes(archive.read(name))
    return receipt


import csv, json, os, re, subprocess, sys, zipfile
from pathlib import Path
from google.colab import drive

def run(command, cwd=None, check=True):
    command = [str(x) for x in command]
    print("+", " ".join(command[:10]), flush=True)
    keep_log("+ " + " ".join(command) + "\n")
    process = subprocess.Popen(command, cwd=cwd, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="", flush=True)
        keep_log(line)
    result = process.wait()
    keep_log(f"[exit {result}]\n")
    if result and check:
        raise RuntimeError("명령 실패: 위 오류를 확인하세요. 결과를 성공으로 처리하지 않습니다.")
    return result

def keep_log(text):
    pass                     # the one-run gold notebook replaces this with a log file on Drive

if N_IMAGES != 240 or not re.fullmatch(r"[A-Za-z0-9_-]+", RUN_TAG):
    raise ValueError("240장 평가 설정을 유지하세요.")

# GPU를 Drive·pip보다 먼저 확인합니다. 런타임이 잘못 잡혔으면 몇 초 만에 멈추고, 지금 연결된 GPU를 알려 줍니다.
try:
    probe = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                           capture_output=True, text=True)
    GPU_NOW = probe.stdout.strip().splitlines()[0] if probe.returncode == 0 and probe.stdout.strip() else ""
except OSError:
    GPU_NOW = ""
GPU_NOW = GPU_NOW or "GPU 없음(CPU 런타임)"
found = re.search(r"(\d+) MiB", GPU_NOW)
if not found or int(found.group(1)) < 20 * 1024:
    raise RuntimeError(f"지금 연결된 GPU: {GPU_NOW}. VRAM 20 GiB 이상인 L4 또는 A100이 필요합니다(양자화로 우회하지 않음). "
                       "런타임 > 런타임 유형 변경에서 L4를 고르고 저장한 다음, 런타임 > 런타임 연결 해제 및 삭제를 누르고 "
                       "다시 모두 실행하세요. L4를 골랐는데도 T4나 GPU 없음으로 나오면 컴퓨팅 단위가 남아 있는지 확인하세요.")
print("GPU:", GPU_NOW)

drive.mount("/content/drive", force_remount=False)
DRIVE = Path("/content/drive/MyDrive")
REPO_DIR = Path("/content/insplad_agent_sft_once")
def verify_code():
    extract_source_bundle(SOURCE_ZIP_B64, SOURCE_ZIP_SHA256, REPO_DIR, EXECUTION_COMMIT)
verify_code()
run([sys.executable, "-m", "pip", "install", "-q", "transformers==5.14.0", "accelerate==1.14.0", "peft>=0.18",
     "safetensors", "sentencepiece==0.2.2", "numpy>=1.26", "pillow>=10", "pandas>=2.2", "tqdm>=4.66", "timm"])
# Colab ships torchao 0.10; peft refuses to build LoRA layers when an old torchao is importable.
# This run never uses torchao (BF16, no quantization), so remove it instead of upgrading torch-coupled code.
run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
if subprocess.run([sys.executable, "-c", "import torchao"], capture_output=True).returncode == 0:
    raise RuntimeError("torchao를 제거하지 못했습니다. 런타임을 다시 시작한 뒤 모두 실행하세요.")
import torch
if not torch.cuda.is_available() or torch.cuda.get_device_properties(0).total_memory < 20 * 1024**3:
    raise RuntimeError(f"torch가 GPU를 쓰지 못합니다(nvidia-smi: {GPU_NOW}, torch {torch.__version__}, "
                       f"CUDA 사용 가능: {torch.cuda.is_available()}). VRAM 20 GiB 이상 L4/A100이 필요하고 "
                       "양자화로 우회하지 않습니다. 런타임을 다시 시작한 뒤 모두 실행하세요.")
print("GPU:", torch.cuda.get_device_name(0), "source:", EXECUTION_COMMIT)
DEV_DIR = DRIVE / DEV_FOLDER
summary = json.loads((DEV_DIR / "build_summary.json").read_text(encoding="utf-8"))
if summary.get("quality_status") != "FROZEN":
    raise RuntimeError("기존 dev 세트가 FROZEN이 아닙니다.")
DEV_QA = (DEV_DIR / summary["file"]).resolve()
if not DEV_QA.is_relative_to(DEV_DIR.resolve()) or not DEV_QA.is_file():
    raise ValueError("dev QA 경로를 확인하세요.")

def find_zip(name, size, override):
    if override:
        p = Path(override)
        if not p.is_file() or p.stat().st_size != size:
            raise ValueError(f"ZIP 경로/크기 불일치: {p}")
        return p
    for p in [DRIVE / name, DRIVE / "lvlm_insplad_data" / name]:
        if p.is_file() and p.stat().st_size == size:
            return p
    matches = [p for p in DRIVE.rglob(name) if p.is_file() and p.stat().st_size == size]
    if len(matches) != 1:
        raise RuntimeError(f"{name}: 후보 {len(matches)}개. 설정 셀에 정확한 경로를 입력하세요.")
    return matches[0]
SUP_ZIP = find_zip("supervised_fault_classification.zip", 874_753_167, SUP_ZIP_PATH)
UNSUP_ZIP = find_zip("unsupervised_anomaly_detection.zip", 1_170_967_719, UNSUP_ZIP_PATH)
REFS_CSV = DEV_DIR / "refs/retrieved_refs.csv"
has_diffdate = False
if REFS_CSV.is_file():
    with REFS_CSV.open(encoding="utf-8-sig", newline="") as stream:
        has_diffdate = "retrieved_diffdate" in (csv.DictReader(stream).fieldnames or [])
if not has_diffdate:
    prepared = DRIVE / "lvlm_insplad_results" / f"agent_tool_pilot_reference_{RUN_TAG}"
    run([sys.executable, REPO_DIR / "scripts/build_retrieved_refs.py", "--qa", DEV_QA,
         "--sup-zip", SUP_ZIP, "--unsup-zip", UNSUP_ZIP, "--out-dir", prepared, "--device", "cuda"])
    REFS_CSV = prepared / "retrieved_refs.csv"
RESULTS = DRIVE / "lvlm_insplad_results"
DATA_DIR = RESULTS / f"agent_sft_data_pc{PER_CLASS}_{RUN_TAG}"
TRAIN_DIR = RESULTS / f"agent_sft_lora_qwen3_5_4b_{RUN_TAG}"
BASE_RUN = RESULTS / f"agent_tool_pilot_dev{N_IMAGES}_qwen3_5_4b_base_{RUN_TAG}"
SFT_RUN = RESULTS / f"agent_tool_pilot_dev{N_IMAGES}_qwen3_5_4b_lora_{RUN_TAG}"
print("학습 데이터:", DATA_DIR, "\n어댑터:", TRAIN_DIR, "\n평가:", BASE_RUN, SFT_RUN)

# 박스 표시가 끝났는지 먼저 확인합니다. 덜 끝난 사진이 있으면 학습을 시작하지 않습니다.
import hashlib
LABEL_DIRS = [RESULTS / name for name in LABEL_RUNS]
for folder, expected in zip(LABEL_DIRS, LABEL_RUNS.values()):
    if not (folder / "manifest.json").is_file() or not (folder / "annotations.json").is_file():
        raise RuntimeError(f"박스 표시 결과가 없습니다: {folder}. 표시 Colab을 먼저 끝내세요.")
    targets = json.loads((folder / "manifest.json").read_text(encoding="utf-8"))["targets"]
    records = json.loads((folder / "annotations.json").read_text(encoding="utf-8"))["records"]
    done = sum(records.get(t["core_id"], {}).get("status") in ("boxed", "unlocatable", "disputed") for t in targets)
    print(f"{folder.name}: 표시 {done}/{len(targets)}장")
    if len(targets) != expected or done != len(targets):
        raise RuntimeError(f"{folder.name}: 표시가 {done}/{len(targets)}장입니다. 표시 Colab에서 남은 사진을 끝낸 뒤 "
                           "이 노트북을 다시 모두 실행하세요. 방금 끝냈다면 Drive 반영까지 1~2분 기다리세요.")
if hashlib.sha256((LABEL_DIRS[0] / "annotations.json").read_bytes()).hexdigest() != FIRST_LABELS_SHA256:
    print("경고: 300장 표시가 2026-11-16에 확인한 파일과 다릅니다(나중에 고친 사진이 있음). 학습은 계속하고, "
          "쓴 표시 파일은 결과 ZIP에 들어갑니다.")
PREVIOUS = {"학습 전 (1차 실행)": RESULTS / f"agent_tool_pilot_dev{N_IMAGES}_qwen3_5_4b_base_sft_v1",
            "lora sft_v1 (자동 박스)": RESULTS / f"agent_tool_pilot_dev{N_IMAGES}_qwen3_5_4b_lora_sft_v1"}

# 시험(gold) 채점 입력을 학습 전에 준비합니다: FROZEN gold QA, 위치 정답 133장(이 노트북에 들어 있음), 정상 참조.
import base64, zlib
GOLD_QA_DIR = DRIVE / GOLD_QA_FOLDER
gold_summary = json.loads((GOLD_QA_DIR / "build_summary.json").read_text(encoding="utf-8"))
if gold_summary.get("quality_status") != "FROZEN" or gold_summary.get("final_master_images") != 246:
    raise RuntimeError(f"FROZEN gold 246장 QA가 필요합니다: {GOLD_QA_DIR}")
GOLD_MASTER = sorted(GOLD_QA_DIR.glob("master_qa_*.jsonl"))[-1]
GOLD_ROOT = RESULTS / f"agent_sft_gold_{RUN_TAG}"
(GOLD_ROOT / "inputs").mkdir(parents=True, exist_ok=True)
LOC_QA = GOLD_ROOT / "inputs/localization_mmad_qa_133.jsonl"
loc_bytes = zlib.decompress(base64.b64decode(LOC_QA_B64))
if hashlib.sha256(loc_bytes).hexdigest() != LOC_QA_SHA256:
    raise RuntimeError("노트북에 들어 있는 위치 정답 파일이 손상됐습니다. 노트북을 다시 받으세요.")
if not LOC_QA.is_file() or LOC_QA.read_bytes() != loc_bytes:
    LOC_QA.write_bytes(loc_bytes)
gold_images = {json.loads(line)["image"] for line in GOLD_MASTER.read_text(encoding="utf-8").splitlines() if line.strip()}
def covers_gold(path):
    if not path.is_file():
        return False
    with path.open(encoding="utf-8-sig", newline="") as stream:
        found = {r["image"]: r.get("retrieved_diffdate") for r in csv.DictReader(stream)}
    return all(found.get(image) for image in gold_images)
GOLD_REFS = next((p for p in (RESULTS / "gold_v2_matched_call_v1/refs_gold/retrieved_refs.csv",
                              RESULTS / "gold_v2_main_rerun/refs_gold/retrieved_refs.csv",
                              GOLD_ROOT / "inputs/refs_gold/retrieved_refs.csv") if covers_gold(p)), None)
if GOLD_REFS is None:
    run([sys.executable, REPO_DIR / "scripts/build_retrieved_refs.py", "--qa", GOLD_MASTER, "--sup-zip", SUP_ZIP,
         "--unsup-zip", UNSUP_ZIP, "--out-dir", GOLD_ROOT / "inputs/refs_gold", "--device", "cuda"])
    GOLD_REFS = GOLD_ROOT / "inputs/refs_gold/retrieved_refs.csv"
V1_ADAPTER = RESULTS / "agent_sft_lora_qwen3_5_4b_sft_v1/adapter"
V1_TRAIN_QA = RESULTS / f"agent_sft_data_pc{PER_CLASS}_sft_v1/sft_train_qa.jsonl"
HAS_V1 = (V1_ADAPTER / "adapter_model.safetensors").is_file() and (V1_ADAPTER / "adapter_config.json").is_file()
GOLD_RUNS = {"gold_lora": GOLD_ROOT / f"agent_sft_gold_eval_lora_{RUN_TAG}",
             "gold_base": GOLD_ROOT / "agent_sft_gold_eval_base",
             "gold_v1": GOLD_ROOT / "agent_sft_gold_eval_lora_sft_v1"}
# dev240 goes to its own folder: SFT_RUN may hold the four-condition run of 2026-11-17 that stopped midway, and a
# run made by other source cannot be resumed by this runner. That folder is left as it is and zipped with the rest.
DEV_RUN = RESULTS / f"agent_tool_pilot_dev{N_IMAGES}_qwen3_5_4b_lora_{RUN_TAG}_two_conditions"
CLASSIFIER_RUN = GOLD_ROOT / "sft_label_classifier_baseline"
print("gold 참조:", GOLD_REFS)
print("1차 모델:", "있음" if HAS_V1 else "Drive에 없음. 1차 모델 시험 채점은 건너뜁니다.")
print("이미 끝난 학습:", "있음. 학습은 건너뜁니다." if (TRAIN_DIR / "adapter/adapter_model.safetensors").is_file()
      else "없음. 학습부터 합니다(6~7시간).")

# 로그, 결과 ZIP, 채점 실행 도우미입니다. 결과 ZIP은 단계가 끝날 때마다 다시 씁니다.
import io, shutil, time
LOG_DIR = GOLD_ROOT / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)
RUN_LOG = LOG_DIR / time.strftime("run_%Y%m%d_%H%M%S.log")
def keep_log(text):
    try:
        with RUN_LOG.open("a", encoding="utf-8") as stream:
            stream.write(text)
    except OSError:
        pass
keep_log(f"notebook source {EXECUTION_COMMIT}, GPU {GPU_NOW}\n")
for name, command in (("pip_freeze.txt", [sys.executable, "-m", "pip", "freeze"]), ("nvidia_smi.txt", ["nvidia-smi"])):
    try:
        (LOG_DIR / name).write_text(subprocess.run(command, capture_output=True, text=True).stdout, encoding="utf-8")
    except OSError:
        pass
EXIT_CODES, EARLIER = {}, {}
ARCHIVE = RESULTS / f"agent_sft_{RUN_TAG}_gold_results.zip"
TRAINED = {"gold_lora": True, "gold_base": False, "gold_v1": True, "dev_lora": True}

def receipt(root, name):
    p = root / name
    return json.loads(p.read_text(encoding="utf-8")) if p.is_file() else {}

def gold_run(key, extra=()):
    """한 모델을 시험(gold) 문제로 채점합니다. 다른 버전으로 시작한 폴더는 이어 갈 수 없어서, 그대로 두고 새 폴더를 씁니다."""
    command = [*GOLD_RUNNER, *extra, "--out-dir", GOLD_RUNS[key]]
    if run([*command, "--inputs-only"], check=False) == 3:
        EARLIER[f"{key}_earlier_run"] = GOLD_RUNS[key]
        GOLD_RUNS[key] = GOLD_RUNS[key].with_name(f"{GOLD_RUNS[key].name}_{EXECUTION_COMMIT[:7]}")
        command[-1] = GOLD_RUNS[key]
        print("이 폴더에는 다른 버전으로 시작한 채점이 있어 새 폴더에서 다시 합니다:", GOLD_RUNS[key].name)
    return run(command, check=False)

def verified_runs():
    """이번 노트북의 소스로 끝까지 돈 단계."""
    found = {}
    for key, root in {**GOLD_RUNS, "dev_lora": DEV_RUN}.items():
        if key not in EXIT_CODES:
            continue
        # If the controls stopped midway, the main step's own receipt is the copy the runner kept before them.
        kept = root / "main_conditions_before_controls"
        now = receipt(root, "run_status.json")
        source = root if now.get("complete") or not (kept / "run_status.json").is_file() else kept
        status, meta = receipt(source, "run_status.json"), receipt(source, "run_metadata.json")
        found[key] = bool(EXIT_CODES[key] == 0 and status.get("complete") and meta.get("execution") == "real_hf"
                          and status.get("run_id") == meta.get("run_id")
                          and meta.get("benchmark_commit") == EXECUTION_COMMIT and meta.get("training") is TRAINED[key])
    if "classifier" in EXIT_CODES:
        status, meta = (receipt(CLASSIFIER_RUN, name) for name in ("run_status.json", "run_metadata.json"))
        found["classifier"] = bool(EXIT_CODES["classifier"] == 0 and status.get("complete")
                                   and status.get("run_id") == meta.get("run_id") and meta.get("execution") == "real_dinov2")
    if "gold_controls" in EXIT_CODES:
        status, meta = (receipt(GOLD_RUNS["gold_lora"], name) for name in ("run_status.json", "run_metadata.json"))
        found["gold_controls"] = bool(EXIT_CODES["gold_controls"] == 0 and status.get("complete")
                                      and status.get("run_id") == meta.get("run_id") and meta.get("controls") is True
                                      and len(status.get("eligible_policies", [])) == 4)
    return found

def preview_images():
    """시험 사진 246장의 축소본(긴 변 640px)을 한 번 만들어 둡니다. 그림을 그릴 때 원본 ZIP을 다시 받지 않아도 됩니다."""
    from PIL import Image
    folder = GOLD_ROOT / "inputs/preview"
    if (folder / "index.csv").is_file():
        return
    folder.mkdir(parents=True, exist_ok=True)
    cores = {}
    for line in GOLD_MASTER.read_text(encoding="utf-8").splitlines():
        if line.strip():
            cores[json.loads(line)["image"]] = json.loads(line)["core_id"]
    rows = []
    with zipfile.ZipFile(SUP_ZIP) as archive:
        names = {n.replace("\\", "/").lstrip("/"): n for n in archive.namelist()}
        for member in sorted(cores):
            data = archive.read(names[member])
            name = hashlib.sha256(data).hexdigest()[:16] + ".jpg"
            with Image.open(io.BytesIO(data)) as picture:
                width, height = picture.size
                small = picture.convert("RGB")
                small.thumbnail((640, 640))
                small.save(folder / name, "JPEG", quality=80)
            rows.append(dict(core_id=cores[member], image=member, file=name, width=width, height=height))
    with (folder / "index.csv").open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)

def save_zip():
    """Drive에 지금 있는 결과로 ZIP을 다시 씁니다. 실패해도 채점은 계속합니다."""
    try:
        try:
            preview_images()
        except Exception as exc:
            print("시험 사진 축소본은 만들지 못했습니다(결과에는 영향 없음):", exc)
        summary = RESULTS / f"agent_sft_{RUN_TAG}_gold_execution.json"
        summary.write_text(json.dumps(dict(
            exit_codes=EXIT_CODES, verified=verified_runs(), execution_commit=EXECUTION_COMMIT, gpu=GPU_NOW,
            written=time.strftime("%Y-%m-%d %H:%M:%S"), gold_refs=str(GOLD_REFS),
            train_qa=[str(p) for p in globals().get("TRAIN_QAS", [])], sft_v1_adapter_found=HAS_V1,
            training_source_commit=receipt(TRAIN_DIR, "train_metadata.json").get("benchmark_commit"),
            run_folders={k: p.name for k, p in {**GOLD_RUNS, "dev_lora": DEV_RUN, "classifier": CLASSIFIER_RUN}.items()},
            earlier_runs={k: p.name for k, p in EARLIER.items()},
            earlier_dev240_run_found=SFT_RUN.is_dir()), indent=2, ensure_ascii=False), encoding="utf-8")
        partial = ARCHIVE.with_name(ARCHIVE.name + ".partial")
        with zipfile.ZipFile(partial, "w", zipfile.ZIP_DEFLATED) as z:
            z.write(summary, "notebook_execution.json")
            z.write(GOLD_REFS, "gold_inputs/retrieved_refs.csv")
            for folder in LABEL_DIRS:                      # the exact label files this run trained on
                for name in ("manifest.json", "annotations.json"):
                    z.write(folder / name, f"labels/{folder.name}/{name}")
            # SFT_RUN: the four-condition dev240 run of 2026-11-17 that stopped midway; its saved answers are kept.
            # EARLIER: gold folders started by another notebook version, kept as they are.
            for label, root in (("gold_inputs", GOLD_ROOT / "inputs"), ("logs", LOG_DIR), ("data", DATA_DIR),
                                ("train", TRAIN_DIR), *GOLD_RUNS.items(), ("classifier", CLASSIFIER_RUN),
                                ("dev_lora", DEV_RUN),
                                ("dev_lora_earlier_four_condition_run", SFT_RUN), *EARLIER.items()):
                if not root.is_dir():
                    continue
                for p in sorted(root.rglob("*")):
                    rel = p.relative_to(root)
                    private = any(part.startswith("_") for part in rel.parts) or rel.parts[0] == "checkpoint"
                    if p.is_file() and not private and p.suffix != ".safetensors":
                        z.write(p, f"{label}/{rel.as_posix()}")
        partial.replace(ARCHIVE)
        print("결과 ZIP 갱신:", ARCHIVE.name, f"{ARCHIVE.stat().st_size / 1e6:.1f} MB")
    except Exception as exc:
        print("결과 ZIP을 갱신하지 못했습니다(채점은 계속하고, 결과 폴더는 Drive에 그대로 있습니다):", exc)


In [ ]:
#@title 2. 학습 데이터 만들기 (사람 박스·궤적)
verify_code()
run([sys.executable, REPO_DIR / "scripts/build_agent_sft_data.py", "--dev-qa", DEV_QA,
     "--sup-zip", SUP_ZIP, "--unsup-zip", UNSUP_ZIP, "--out-dir", DATA_DIR,
     "--per-class", PER_CLASS, "--device", "cuda", "--add-polymer-corrosion", "--human-boxes", *LABEL_DIRS])
built = json.loads((DATA_DIR / "build_summary.json").read_text(encoding="utf-8"))
if built["protocol"] != "agent_sft_train_v2_human_boxes" or built["trajectories"].get("crop") != sum(
        r["status"].get("boxed", 0) for r in built["human_box_runs"]):
    raise RuntimeError("사람 박스로 만든 학습 데이터가 아닙니다. 결과를 쓰지 마세요.")
print(json.dumps({k: built[k] for k in ("images", "samples", "trajectories", "q3_cells", "distinct_crop_boxes",
                                        "grown_small_boxes", "unlocated_photos_left_out")}, ensure_ascii=False))

# 학습을 시작하기 전에 시험 채점 입력을 끝까지 점검합니다(모델 호출 없음). 여기서 멈추면 학습 시간을 쓰지 않습니다.
TRAIN_QAS = [DATA_DIR / "sft_train_qa.jsonl"] + ([V1_TRAIN_QA] if V1_TRAIN_QA.is_file() else [])
GOLD_COMMON = ["--qa-dir", GOLD_QA_DIR, "--loc-qa", LOC_QA, "--refs-csv", GOLD_REFS, "--sup-zip", SUP_ZIP,
               "--unsup-zip", UNSUP_ZIP, "--train-qa", *TRAIN_QAS, "--max-calls", 3]
GOLD_RUNNER = [sys.executable, REPO_DIR / "scripts/run_agent_sft_gold_eval.py", *GOLD_COMMON]
# A folder name of its own: this check writes nothing and must not be compared with a saved run.
run([*GOLD_RUNNER, "--out-dir", GOLD_ROOT / "agent_sft_gold_eval_input_check", "--inputs-only"])


In [ ]:
#@title 3. Qwen3.5-4B LoRA 학습 (BF16)
verify_code()
run([sys.executable, REPO_DIR / "scripts/train_agent_sft_lora.py", "--data-dir", DATA_DIR,
     "--out-dir", TRAIN_DIR, "--epochs", EPOCHS, "--lr", LEARNING_RATE])


In [ ]:
#@title 4. 시험(gold) 채점 — 2차 모델 (주 결과)
verify_code()
EXIT_CODES["gold_lora"] = gold_run("gold_lora", ["--adapter-dir", TRAIN_DIR / "adapter"])
print(EXIT_CODES)
save_zip()


In [ ]:
#@title 5. 작은 분류기 비교 — 같은 라벨 660장, DINOv2 특징 + 선형 분류기
# LVLM 대신 작은 전용 분류기를 같은 학습 사진으로 학습해 같은 시험 문제의 Q1·Q2를 풉니다(dev240도 같이). 위치는 답하지 못합니다.
# 뽑은 특징은 결과 ZIP에 들어가므로, 다른 작은 분류기는 나중에 GPU 없이 시험해 볼 수 있습니다.
verify_code()
if RUN_CLASSIFIER_BASELINE:
    EXIT_CODES["classifier"] = run([sys.executable, REPO_DIR / "scripts/run_sft_label_classifier_baseline.py",
                                    "--train-qa", DATA_DIR / "sft_train_qa.jsonl", "--overlap-train-qa", *TRAIN_QAS,
                                    "--qa-dir", GOLD_QA_DIR, "--sup-zip", SUP_ZIP, "--dev-qa", DEV_QA,
                                    "--n-images", N_IMAGES, "--device", "cuda", "--out-dir", CLASSIFIER_RUN], check=False)
else:
    print("작은 분류기 비교는 건너뜁니다.")
print(EXIT_CODES)
save_zip()


In [ ]:
#@title 6. 시험(gold) 채점 — 학습 전 모델
verify_code()
if RUN_GOLD_BASE:
    EXIT_CODES["gold_base"] = gold_run("gold_base")
else:
    print("학습 전 모델 시험 채점은 건너뜁니다.")
print(EXIT_CODES)
save_zip()


In [ ]:
#@title 7. dev240 평가 — 2차 모델 (한 번에 답, 도구 선택)
verify_code()
if RUN_DEV240:
    EXIT_CODES["dev_lora"] = run([sys.executable, REPO_DIR / "scripts/run_agent_tool_pilot.py", "--dev-qa", DEV_QA,
                                  "--refs-csv", REFS_CSV, "--sup-zip", SUP_ZIP, "--unsup-zip", UNSUP_ZIP,
                                  "--model", "qwen3_5_4b", "--n-images", N_IMAGES, "--max-calls", 3,
                                  "--policies", "direct_reference", "adaptive_tools",
                                  "--out-dir", DEV_RUN, "--adapter-dir", TRAIN_DIR / "adapter"], check=False)
else:
    print("dev240 평가는 건너뜁니다.")
print(EXIT_CODES)
save_zip()


In [ ]:
#@title 8. 시험(gold) 채점 — 1차 모델 (Drive에 있을 때만)
verify_code()
if RUN_GOLD_SFT_V1 and HAS_V1:
    EXIT_CODES["gold_v1"] = gold_run("gold_v1", ["--adapter-dir", V1_ADAPTER])
else:
    print("1차 모델 시험 채점은 건너뜁니다.")
print(EXIT_CODES)
save_zip()


In [ ]:
#@title 9. 시험(gold) 채점 — 2차 모델의 대조 조건 (가운데 고정 확대, 같은 횟수 다시 묻기)
# 4번에서 저장한 답은 그대로 다시 쓰고, 대조 조건 두 개만 새로 묻습니다. 4번의 표는 폴더 안에 따로 보존됩니다.
verify_code()
if RUN_GOLD_CONTROLS and EXIT_CODES.get("gold_lora") == 0:
    EXIT_CODES["gold_controls"] = gold_run("gold_lora", ["--adapter-dir", TRAIN_DIR / "adapter", "--controls"])
else:
    print("대조 조건은 건너뜁니다(스위치를 껐거나 4번 채점이 끝나지 않았습니다).")
print(EXIT_CODES)
save_zip()


In [ ]:
#@title 10. 표와 결과 ZIP
save_zip()                                           # the ZIP comes first and never depends on the tables below
VERIFIED = verified_runs()
for key in ("gold_lora", "classifier", "gold_base", "dev_lora", "gold_v1", "gold_controls"):
    if key not in EXIT_CODES:
        print(key, "건너뜀")
        continue
    root = {**GOLD_RUNS, "dev_lora": DEV_RUN, "classifier": CLASSIFIER_RUN, "gold_controls": GOLD_RUNS["gold_lora"]}[key]
    print(key, "완료 확인" if VERIFIED.get(key) else "실행 오류 또는 완료 미확인",
          json.dumps(receipt(root, "run_status.json"), ensure_ascii=False))
try:
    import pandas as pd
    def boards(pairs):
        found = []
        for label, root in pairs:
            board = root / "leaderboard.csv"
            if board.is_file() and board.stat().st_size:
                found.append(pd.read_csv(board).assign(model=label))
        return pd.concat(found) if found else None
    NAMES = {"gold_lora": f"lora {RUN_TAG} (사람 박스)", "gold_base": "학습 전", "gold_v1": "lora sft_v1 (자동 박스)"}
    table = boards((NAMES[key], root) for key, root in GOLD_RUNS.items())
    if table is not None:
        cols = ["model", "subset", "policy", "images", "q1_anomaly_f1", "q1_balanced_accuracy", "q2_defect_macro_f1",
                "joint_accuracy", "q3_presence_balanced_accuracy", "q3_position_hit", "q3_anomalous_location_labels",
                "q3_always_center_hit", "q1_q2_q3_location_joint", "q3_crop_covers_defect", "q3_crops_on_located",
                "center60_covers_defect", "crop_distinct_boxes", "invalid_episodes", "calls"]
        print("시험(gold) 결과. 주 결과는 subset=frame_disjoint(학습 사진과 같은 드론 사진을 뺀 사진), 2차 모델, direct_reference 행입니다.")
        print("fixed_evidence와 matched_reask는 대조 조건입니다(2차 모델만).")
        display(table[[c for c in cols if c in table]])
    table = boards([("DINOv2 특징 + 선형 분류기", CLASSIFIER_RUN)])
    if table is not None:
        cols = ["model", "eval_set", "subset", "images", "q1_anomaly_f1", "q1_balanced_accuracy", "q2_defect_macro_f1",
                "joint_accuracy", "false_defect_rate", "q1_tp", "q1_fp", "q1_tn", "q1_fn"]
        print("작은 분류기 결과(같은 학습 사진 660장, Q1·Q2만). gold의 subset=frame_disjoint 행을 위 표의 같은 subset과 비교합니다.")
        display(table[[c for c in cols if c in table]])
    table = boards(((NAMES["gold_lora"], DEV_RUN), *PREVIOUS.items()))
    if table is not None:
        cols = ["model", "policy", "q1_balanced_accuracy", "q2_defect_macro_f1", "q2_invalid_rate", "joint_accuracy",
                "false_defect_rate", "q3_presence_balanced_accuracy", "q3_normal_false_region_rate", "invalid_episodes", "calls"]
        print("dev240 결과. 지난 실행의 행은 Drive에 남은 결과를 읽은 것입니다(이번에 다시 돌리지 않음).")
        display(table[[c for c in cols if c in table]])
except Exception as exc:
    print("표를 그리지 못했습니다(결과 ZIP에는 영향 없음):", exc)
train_meta = receipt(TRAIN_DIR, "train_metadata.json")
print("학습:", {k: train_meta.get(k) for k in ("optimizer_steps", "val_loss_before", "val_loss_after")})
if train_meta.get("benchmark_commit") != EXECUTION_COMMIT:
    print("학습은 이전 실행(소스", str(train_meta.get("benchmark_commit"))[:7] + ")이 Drive에 남긴 어댑터를 그대로 썼습니다.")
print("결과 ZIP은 Drive에 저장됐습니다:", ARCHIVE)
print("모두 완료" if VERIFIED and all(VERIFIED.values()) else "일부 미완료 또는 건너뜀(위 목록 확인)")
try:
    from google.colab import files
    local = Path("/content") / ARCHIVE.name             # a local copy, so the download does not depend on Drive
    shutil.copyfile(ARCHIVE, local)
    files.download(str(local))
except Exception as exc:
    print("브라우저 다운로드는 건너뜁니다. Drive에 있는 ZIP을 쓰세요:", exc)
try:
    drive.flush_and_unmount()                           # push everything to Drive before the runtime can go away
    print("Drive 저장을 마쳤습니다. 이제 런타임을 끊어도 됩니다.")
except Exception as exc:
    print("Drive 마무리 저장을 확인하지 못했습니다. 몇 분 뒤에 런타임을 끊으세요:", exc)
